# Scrap GlassDoor Reviews for US top500 Companies

code modified from https://scrapfly.io/blog/how-to-scrape-glassdoor/

In [ ]:
import json
from scrapfly import ScrapeApiResponse, ScrapeConfig, ScrapflyClient
import re
import asyncio
from typing import Dict, Optional
import urllib.parse

client = ScrapflyClient(key="    ")

In [2]:
# BASE_CONFIG = {
#     "country": "US",
#     "asp": True,
#     "cookies": {"tldp": "1"},
#     "cost_budget": 56
# }


# def find_hidden_data(result: ScrapeApiResponse) -> dict:
#     """
#     Extract hidden web cache (Apollo Graphql framework) from Glassdoor page HTML
#     It's either in NEXT_DATA script or direct apolloState js variable
#     """
#     # data can be in __NEXT_DATA__ cache
#     data = result.selector.css("script#__NEXT_DATA__::text").get()
#     if data:
#         data = json.loads(data)["props"]["pageProps"]["apolloCache"]
#     else:  # or in direct apolloState cache
#         data = re.findall(r'apolloState":\s*({.+})};', result.content)[0]
#         data = json.loads(data)

#     def _unpack_apollo_data(apollo_data):
#         """
#         Glassdoor uses Apollo GraphQL client and the dataset is a graph of references.
#         This function unpacks the __ref references to actual values.
#         """

#         def resolve_refs(data, root):
#             if isinstance(data, dict):
#                 if "__ref" in data:
#                     return resolve_refs(root[data["__ref"]], root)
#                 else:
#                     return {k: resolve_refs(v, root) for k, v in data.items()}
#             elif isinstance(data, list):
#                 return [resolve_refs(i, root) for i in data]
#             else:
#                 return data

#         return resolve_refs(apollo_data.get("ROOT_QUERY") or apollo_data, apollo_data)

#     return _unpack_apollo_data(data)


# def change_page(url: str, page: int) -> str:
#     """update page number in a glassdoor url"""
#     if re.search(r"_P\d+\.htm", url):
#         new = re.sub(r"(?:_P\d+)*.htm", f"_P{page}.htm", url)
#     else:
#         new = re.sub(".htm", f"_P{page}.htm", url)
#     assert new != url
#     return new


# def parse_reviews(result: ScrapeApiResponse) -> Dict:
#     """parse Glassdoor reviews page for review data"""
#     cache = find_hidden_data(result)
#     reviews = next(v for k, v in cache.items() if k.startswith("employerReviews") and v.get("reviews"))
#     return reviews


# async def scrape_reviews(url: str, max_pages: Optional[int] = None) -> Dict:
#     """Scrape Glassdoor reviews listings from reviews page (with pagination)"""
#     print(f"scraping reviews from {url}")
#     first_page = await client.async_scrape(ScrapeConfig(url=url, **BASE_CONFIG))

#     reviews = parse_reviews(first_page)
#     total_pages = reviews["numberOfPages"]
#     if max_pages and max_pages < total_pages:
#         total_pages = max_pages

#     print(f"scraped first page of reviews of {url}, remaining {total_pages - 1} pages")
#     other_pages = [
#         ScrapeConfig(url=change_page(first_page.context["url"], page=page), **BASE_CONFIG)
#         for page in range(2, total_pages + 1)
#     ]
#     async for result in client.concurrent_scrape(other_pages):
#         reviews["reviews"].extend(parse_reviews(result)["reviews"])
#     print("scraped {} reviews from {} in {} pages".format(len(reviews["reviews"]), url, total_pages))
#     return reviews

# async def run_one(company, id):
#     company = urllib.parse.quote(company)
#     id = urllib.parse.quote(id)
#     url = f"https://www.glassdoor.com/Reviews/{company}-Reviews-E{id}.htm"
#     reviews = await scrape_reviews(
#         url=url,
#         max_pages=2
#     )
#     # print(json.dumps(ebay_jobs_in_US, indent=2))
#     return reviews
#     # with open("ebay_reviews.json", "w") as f:
#     #     json.dump(ebay_jobs_in_US, f, indent=2)


In [3]:
BASE_CONFIG = {
    "country": "US",
    "asp": True,
    "cookies": {"tldp": "1"},
    "cost_budget": 80
}


def find_hidden_data(result: ScrapeApiResponse) -> dict:
    """
    Extract hidden web cache (Apollo Graphql framework) from Glassdoor page HTML
    It's either in NEXT_DATA script or direct apolloState js variable
    """
    # data can be in __NEXT_DATA__ cache
    data = result.selector.css("script#__NEXT_DATA__::text").get()
    if data:
        data = json.loads(data)["props"]["pageProps"]["apolloCache"]
    else:  # or in direct apolloState cache
        data = re.findall(r'apolloState":\s*({.+})};', result.content)[0]
        data = json.loads(data)

    def _unpack_apollo_data(apollo_data):
        """
        Glassdoor uses Apollo GraphQL client and the dataset is a graph of references.
        This function unpacks the __ref references to actual values.
        """

        def resolve_refs(data, root):
            if isinstance(data, dict):
                if "__ref" in data:
                    return resolve_refs(root[data["__ref"]], root)
                else:
                    return {k: resolve_refs(v, root) for k, v in data.items()}
            elif isinstance(data, list):
                return [resolve_refs(i, root) for i in data]
            else:
                return data

        return resolve_refs(apollo_data.get("ROOT_QUERY") or apollo_data, apollo_data)

    return _unpack_apollo_data(data)


def change_page(url: str, page: int) -> str:
    """update page number in a glassdoor url"""
    if re.search(r"_P\d+\.htm", url):
        new = re.sub(r"(?:_P\d+)*.htm", f"_P{page}.htm", url)
    else:
        new = re.sub(".htm", f"_P{page}.htm", url)
    assert new != url
    return new


def parse_reviews(result: ScrapeApiResponse) -> Dict:
    """parse Glassdoor reviews page for review data"""
    cache = find_hidden_data(result)
    reviews = next(v for k, v in cache.items() if k.startswith("employerReviews") and v.get("reviews"))
    print(type(reviews))
    return reviews


def scrape_reviews(url: str, max_pages: Optional[int] = None) -> Dict:
    """Scrape Glassdoor reviews listings from reviews page (with pagination)"""
    print(f"scraping reviews from {url}")
    first_page = client.scrape(ScrapeConfig(url=url, **BASE_CONFIG))

    reviews = parse_reviews(first_page)
    total_pages = reviews["numberOfPages"]
    if max_pages and max_pages < total_pages:
        total_pages = max_pages

    print(f"scraped first page of reviews of {url}, remaining {total_pages - 1} pages")
    for page in range(2, total_pages + 1):
        other_page = client.scrape(ScrapeConfig(url=change_page(first_page.context["url"], page), **BASE_CONFIG))
        reviews["reviews"].extend(parse_reviews(other_page)["reviews"])
    print("scraped {} reviews from {} in {} pages".format(len(reviews["reviews"]), url, total_pages))
    return reviews

def run_one(company, id):
    company = urllib.parse.quote(company)
    id = urllib.parse.quote(id)
    url = f"https://www.glassdoor.com/Reviews/{company}-Reviews-E{id}.htm"
    reviews = scrape_reviews(
        url=url,
        max_pages=2
    )
    # print(json.dumps(ebay_jobs_in_US, indent=2))
    return reviews
    # with open("ebay_reviews.json", "w") as f:
    #     json.dump(ebay_jobs_in_US, f, indent=2)

In [4]:
with open("top500_reviews.json", "r") as f:
    companies = json.load(f)

print("{} companies to scrape".format(len(companies)))

# print(companies[0])
# ct = 0
# ids = set()
# for company in companies:
#     if "employerId" in company:
        
#         if company["employerId"] in ids and company["employerId"] != "0":
#             print(company["company"], company["employerId"])
#             ct += 1
#         ids.add(company["employerId"])
# print(ct)
# print(len(ids))

# print(len(companies))

500 companies to scrape


In [5]:

failed = []
repeat = []
results = []
ids = set()
succ_ct, exist_ct = 0, 0

for i in range(len(companies)):
    try:
        company = companies[i]["company"]
        employerId = companies[i]["employerId"]
        if employerId in ids:
            repeat.append((company, employerId))
            continue
        ids.add(employerId)
        if "reviews" not in companies[i]:
            result = run_one(company, employerId)
            # print(result)
            # results.append(result)
            companies[i]["reviews"] = result
            succ_ct += 1
        else:
            print("{}: already have reviews, skiped".format(company))
        exist_ct += 1
        print(f"finished {company}")
    except Exception as e:
        print(repr(e))
        print(f"failed {company}")
        failed.append((company, employerId))

print("{} newly scraped".format(succ_ct))
print("{} out of {} companies done".format(exist_ct, len(companies)))
print("{} companies has duplicate employerId".format(len(repeat)))
print("{} companies failed".format(len(failed)))

Walmart: already have reviews, skiped
finished Walmart
Amazon.com: already have reviews, skiped
finished Amazon.com
Exxon Mobil: already have reviews, skiped
finished Exxon Mobil
Apple: already have reviews, skiped
finished Apple
UnitedHealth Group: already have reviews, skiped
finished UnitedHealth Group
CVS Health: already have reviews, skiped
finished CVS Health
Berkshire Hathaway: already have reviews, skiped
finished Berkshire Hathaway
Alphabet: already have reviews, skiped
finished Alphabet
McKesson: already have reviews, skiped
finished McKesson
Chevron: already have reviews, skiped
finished Chevron
AmerisourceBergen: already have reviews, skiped
finished AmerisourceBergen
Costco Wholesale: already have reviews, skiped
finished Costco Wholesale
Microsoft: already have reviews, skiped
finished Microsoft
Cardinal Health: already have reviews, skiped
finished Cardinal Health
Cigna Group: already have reviews, skiped
finished Cigna Group
Marathon Petroleum: already have reviews, ski

CRITICAL:root:<-- 200 | ERR::SCRAPE::BAD_UPSTREAM_RESPONSE - The website you target respond with an unexpected status code (>400) - The website you scrape respond with: 404 - Not Found.Checkout the related doc: https://scrapfly.io/docs/scrape-api/error/ERR::SCRAPE::BAD_UPSTREAM_RESPONSE


UpstreamHttpClientError('404 ERR::SCRAPE::BAD_UPSTREAM_RESPONSE The website you target respond with an unexpected status code (>400) - The website you scrape respond with: 404 - Not Found', 'ERR::SCRAPE::BAD_UPSTREAM_RESPONSE')
failed Charter Communications
Tyson Foods: already have reviews, skiped
finished Tyson Foods
Deere: already have reviews, skiped
finished Deere
Cisco Systems: already have reviews, skiped
finished Cisco Systems
Nationwide: already have reviews, skiped
finished Nationwide
Allstate: already have reviews, skiped
finished Allstate
Delta Air Lines: already have reviews, skiped
finished Delta Air Lines
Liberty Mutual Insurance Group: already have reviews, skiped
finished Liberty Mutual Insurance Group
TJX: already have reviews, skiped
finished TJX
Progressive: already have reviews, skiped
finished Progressive
American Airlines Group: already have reviews, skiped
finished American Airlines Group
CHS: already have reviews, skiped
finished CHS
Performance Food Group: alr

CRITICAL:root:<-- 200 | ERR::SCRAPE::BAD_UPSTREAM_RESPONSE - The website you target respond with an unexpected status code (>400) - The website you scrape respond with: 404 - Not Found.Checkout the related doc: https://scrapfly.io/docs/scrape-api/error/ERR::SCRAPE::BAD_UPSTREAM_RESPONSE


UpstreamHttpClientError('404 ERR::SCRAPE::BAD_UPSTREAM_RESPONSE The website you target respond with an unexpected status code (>400) - The website you scrape respond with: 404 - Not Found', 'ERR::SCRAPE::BAD_UPSTREAM_RESPONSE')
failed Genworth Financial
scraping reviews from https://www.glassdoor.com/Reviews/EQT-Reviews-E233.htm
<class 'dict'>
scraped first page of reviews of https://www.glassdoor.com/Reviews/EQT-Reviews-E233.htm, remaining 1 pages
<class 'dict'>
scraped 20 reviews from https://www.glassdoor.com/Reviews/EQT-Reviews-E233.htm in 2 pages
finished EQT
scraping reviews from https://www.glassdoor.com/Reviews/Skechers%20U.S.A.-Reviews-E8575.htm
<class 'dict'>
scraped first page of reviews of https://www.glassdoor.com/Reviews/Skechers%20U.S.A.-Reviews-E8575.htm, remaining 1 pages
<class 'dict'>
scraped 20 reviews from https://www.glassdoor.com/Reviews/Skechers%20U.S.A.-Reviews-E8575.htm in 2 pages
finished Skechers U.S.A.
scraping reviews from https://www.glassdoor.com/Reviews

In [6]:
print(failed)
print(repeat)



[('Charter Communications', '0'), ('Berry Global Group', '1065223'), ('Crown Holdings', '3098089'), ('Hormel Foods', '8155409'), ('Masco', '4889482'), ('Westinghouse Air Brake Technologies', '8306659'), ('ARKO', '2607195'), ('Huntington Bancshares', '34794'), ('Genworth Financial', '34925')]
[('Hartford Financial Services Group', '0'), ('PG&E', '0'), ('Goodyear Tire & Rubber', '0'), ('Carrier Global', '0'), ('Bank of New York Mellon', '0'), ('Lincoln National', '0'), ('Principal Financial', '0'), ('Expeditors International of Washington', '0'), ('Automatic Data Processing', '0'), ('Opendoor Technologies', '5038656'), ('Ball', '0'), ('Jacobs Solutions', '0'), ('Laboratory Corp. of America', '0'), ('Fidelity National Information Services', '0'), ('IQVIA Holdings', '6656459'), ("O'Reilly Automotive", '0'), ('Farmers Insurance Exchange', '0'), ('AES', '0'), ('Mutual of Omaha Insurance', '0'), ('APA', '0'), ('VF', '0'), ('TravelCenters of America', '8874'), ('News Corp.', '0'), ('Fifth Thir

In [7]:
# final_results = [results_dict[key] for key in results_dict.keys()]


with open("top500_reviews2.json", "w") as f:
    json.dump(companies, f, indent=2)